In [40]:
import numpy as np
import pandas as pd
from collections import Counter

In [41]:
data = pd.read_csv("BankNote_Authentication.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1372 entries, 0 to 1371
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   variance  1372 non-null   float64
 1   skewness  1372 non-null   float64
 2   curtosis  1372 non-null   float64
 3   entropy   1372 non-null   float64
 4   class     1372 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 53.7 KB


In [42]:
indices = np.random.permutation(len(data)) # gets indices col in random order
data = data.iloc[indices]

train_split = int(len(data) * 0.8)
train_data = data.iloc[:train_split]
test_data = data.iloc[train_split:]


X_train = train_data[["variance", "skewness", "curtosis", "entropy"]]
y_train = train_data["class"]

X_test = test_data[["variance", "skewness", "curtosis", "entropy"]]
y_test = test_data["class"]

In [43]:
def gini(labels):
    count = Counter(labels)
    summition = 0
    
    def probability(count):
        prob = count / len(labels)
        return prob

    def squares(probability):
        return probability ** 2
    
    for value in count.values():
        summition += squares(probability(value))
        
    return 1 - summition

gini(y_train)

0.4932986208346457

In [ ]:
def find_best_split(X_train, y_train):

    best_gini = float("inf")
    best_threshold = None
    best_feature = None

    for feature in X_train.columns:
        for threshold in X_train[feature].unique():

            left_mask = X_train[feature] <= threshold
            right_mask = X_train[feature] > threshold
            
            left_labels = y_train[left_mask]
            right_labels = y_train[right_mask]

            if len(left_labels) == 0 or len(right_labels) == 0:
                continue
            
            n = len(left_labels) + len(right_labels)

            weighted_gini = (
                (len(left_labels) / n) * gini(left_labels)
                +
                (len(right_labels) / n) * gini(right_labels)
            )

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_threshold = threshold
                best_feature = feature

    return best_threshold, best_gini, best_feature

In [ ]:
def build_tree(X, y, depth=0, max_depth=5):

    if len(Counter(y)) == 1 or depth == max_depth:
        return Counter(y).most_common(1)[0][0]

    best_threshold, best_gini, best_feature = find_best_split(X, y)

    left_mask = X[best_feature] <= best_threshold
    right_mask = X[best_feature] > best_threshold

    X_left = X[left_mask]
    X_right = X[right_mask]

    y_left = y[left_mask]
    y_right = y[right_mask]

    left_tree = build_tree(X_left, y_left, depth + 1, max_depth)
    right_tree = build_tree(X_right, y_right, depth + 1, max_depth)

    return {
        "feature": best_feature,
        "threshold": best_threshold,
        "left": left_tree,
        "right": right_tree
    }


tree = build_tree(X_train, y_train)

print(tree)

{'feature': 'variance', 'threshold': np.float64(0.31803), 'left': {'feature': 'skewness', 'threshold': np.float64(6.8017), 'left': {'feature': 'variance', 'threshold': np.float64(-0.46651), 'left': {'feature': 'curtosis', 'threshold': np.float64(6.2109), 'left': 1, 'right': {'feature': 'skewness', 'threshold': np.float64(-4.5566), 'left': 1, 'right': 0}}, 'right': {'feature': 'curtosis', 'threshold': np.float64(4.7749), 'left': {'feature': 'skewness', 'threshold': np.float64(5.0097), 'left': 1, 'right': 0}, 'right': 0}}, 'right': {'feature': 'variance', 'threshold': np.float64(-4.4779), 'left': 1, 'right': 0}}, 'right': {'feature': 'curtosis', 'threshold': np.float64(-4.3882), 'left': {'feature': 'variance', 'threshold': np.float64(2.3917), 'left': 1, 'right': 0}, 'right': {'feature': 'variance', 'threshold': np.float64(1.5904), 'left': {'feature': 'curtosis', 'threshold': np.float64(-2.2726), 'left': {'feature': 'skewness', 'threshold': np.float64(3.644), 'left': 1, 'right': 0}, 'righ

In [46]:
def predict_one(row, tree):

    if not isinstance(tree, dict):
        return tree

    feature = tree["feature"]
    threshold = tree["threshold"]

    if row[feature] <= threshold:
        return predict_one(row, tree["left"])
    else:
        return predict_one(row, tree["right"])

In [48]:
predictions = []

for i in range(len(X_test)):
    prediction = predict_one(X_test.iloc[i], tree)
    predictions.append(prediction)
    
accuracy = np.mean(np.array(predictions) == np.array(y_test))

print("Accuracy:", accuracy)

Accuracy: 0.9745454545454545
